In [1]:
%pip install --quiet python-dotenv pydantic-ai pandas

import os
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()

# Download data files if not already present (e.g. on Colab)
if not Path("data").exists():
    import zipfile, urllib.request
    url = "https://github.com/jsoma/workshop-ai-images-video/raw/main/docs/nicar-2026/05-recipes-data.zip"
    print("Downloading data...")
    urllib.request.urlretrieve(url, "_data.zip")
    with zipfile.ZipFile("_data.zip") as zf:
        zf.extractall("data")
    Path("_data.zip").unlink()
    print("Done!")

# Load API keys: Colab secrets or local .env
try:
    from google.colab import userdata
    os.environ.setdefault("GOOGLE_API_KEY", userdata.get("GOOGLE_API_KEY"))
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


Note: you may need to restart the kernel to use updated packages.


# The full pipeline

## Screen time

Frames → classify → per-subject screen time with percentages. This is the deliverable you'd bring to an editor meeting.


**`recipes/screen-time.py`** — Classify pre-extracted debate frames → per-subject screen time summary.


In [2]:
# Audit trail: every frame gets a row, not a vibe answer.
from pathlib import Path
import mimetypes, re
import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")

MODEL, SECONDS_PER_FRAME = "openai:gpt-4o-mini", 1.0

class FrameClassification(BaseModel):
    primary_subject: str = Field(description="Who is primarily on screen (name or description)")
    scene_type: str = Field(description="'close-up', 'wide shot', 'split screen', 'graphic', 'audience', 'other'")
    confidence: float = Field(ge=0.0, le=1.0)

# --- Step 1: Discover frames ---
frames_dir = DATA / "debate"
frames = sorted(p for p in frames_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(f"Found {len(frames)} frames")

# --- Step 2: Classify each frame ---
agent = Agent(MODEL, output_type=FrameClassification)
rows = []
for fpath in frames:
    num = int(m.group(1)) if (m := re.search(r"(\d+)", fpath.stem)) else 0
    mime, _ = mimetypes.guess_type(str(fpath))
    r = agent.run_sync([
        "This is a frame from a political debate. Identify who is on screen.",
        BinaryContent(data=fpath.read_bytes(), media_type=mime or "image/jpeg"),
    ])
    rows.append({"filename": fpath.name, "frame_number": num, "timestamp_sec": num * SECONDS_PER_FRAME,
                 "primary_subject": r.output.primary_subject, "scene_type": r.output.scene_type,
                 "confidence": r.output.confidence})
print(f"Classified {len(rows)} frames")

# --- Step 3: Build summary ---
df = pd.DataFrame(rows).sort_values("frame_number").reset_index(drop=True)
summary = (
    df.groupby("primary_subject")
    .agg(frames=("primary_subject", "count"),
         total_seconds=("timestamp_sec", lambda x: len(x) * SECONDS_PER_FRAME),
         avg_confidence=("confidence", "mean"))
    .sort_values("total_seconds", ascending=False)
)
summary["pct_of_total"] = (summary["total_seconds"] / summary["total_seconds"].sum() * 100).round(1)
print(summary)

df


Found 30 frames


Classified 30 frames
                                             frames  total_seconds  \
primary_subject                                                      
unknown                                           7            7.0   
Joe Biden                                         5            5.0   
Donald Trump                                      4            4.0   
a political figure                                2            2.0   
A political figure                                1            1.0   
A political figure speaking in a debate           1            1.0   
Not identified                                    1            1.0   
Person at a political debate                      1            1.0   
Political candidates                              1            1.0   
a male politician                                 1            1.0   
a male politician speaking during a debate        1            1.0   
a man speaking at a political debate              1            1.0   

,filename,frame_number,timestamp_sec,primary_subject,scene_type,confidence
0,frame-001.jpg,1,1.0,Political candidates,split screen,0.90
1,frame-002.jpg,2,2.0,unknown,close-up,0.00
2,frame-003.jpg,3,3.0,unknown,close-up,0.00
3,frame-004.jpg,4,4.0,a politician speaking during a debate,close-up,0.90
4,frame-005.jpg,5,5.0,Donald Trump,close-up,0.90
5,frame-006.jpg,6,6.0,Donald Trump,close-up,0.90
6,frame-007.jpg,7,7.0,a male politician,close-up,0.90
7,frame-008.jpg,8,8.0,A political figure speaking in a debate,close-up,0.90
8,frame-009.jpg,9,9.0,Joe Biden,close-up,0.90
9,frame-010.jpg,10,10.0,A political figure,close-up,0.70


## Cost

Before you run a big batch: how much will it cost? Images × tokens × price = receipt.


**`recipes/cost.py`** — Estimate API cost for a batch of images. No API key needed.


In [3]:
# USD per million tokens (Feb 2026)
PRICE_TABLE = {
    "gpt-4o-mini":       (0.15, 0.60),
    "gpt-4o":            (2.50, 10.00),
    "gpt-5-nano":        (0.05, 0.20),
    "gemini-2.5-flash":  (0.10, 0.40),
    "gemini-2.5-pro":    (1.25, 5.00),
    "claude-3-5-haiku":  (0.80, 4.00),
    "claude-3-5-sonnet": (3.00, 15.00),
    "ollama":            (0.00, 0.00),
}

NUM_IMAGES = 500
MODEL = "gpt-4o-mini"
INPUT_TOKENS_PER_IMAGE = 1000
OUTPUT_TOKENS_PER_IMAGE = 200

# --- Calculate ---
input_price, output_price = PRICE_TABLE[MODEL]
total_input = NUM_IMAGES * INPUT_TOKENS_PER_IMAGE
total_output = NUM_IMAGES * OUTPUT_TOKENS_PER_IMAGE
input_cost = (total_input / 1_000_000) * input_price
output_cost = (total_output / 1_000_000) * output_price
total_cost = input_cost + output_cost

print(f"Model:       {MODEL}")
print(f"Images:      {NUM_IMAGES:,}")
print(f"Input cost:  ${input_cost:.4f}  ({total_input:,} tokens @ ${input_price}/M)")
print(f"Output cost: ${output_cost:.4f}  ({total_output:,} tokens @ ${output_price}/M)")
print(f"TOTAL:       ${total_cost:.4f}")


Model:       gpt-4o-mini
Images:      500
Input cost:  $0.0750  (500,000 tokens @ $0.15/M)
Output cost: $0.0600  (100,000 tokens @ $0.6/M)
TOTAL:       $0.1350
